# pandas: loading, cleaning, grouping

**M0 · Toolbox & Submission Craft** — [AI Olympiad Curriculum](https://ai-olympiad-curriculum.netlify.app)

~35 min

## First look at a dataset

A DataFrame is a table with named, typed columns — think of a sheet where every column has a name and a dtype. These calls fit on one screen and should run before anything else:



In [ ]:
import pandas as pd
df = pd.read_csv("train.csv")
df.shape                       # (rows, columns)
df.dtypes                      # dtype per column — watch for "object" that should be numbers
df.head()                      # first 5 rows
df.describe()                  # min/mean/max/quantiles of numeric columns
df.isna().mean()               # share of missing values per column
df["target"].value_counts()    # class balance — is one class 90% of the data?

Two more for the toolbox: `df.info()` (memory use and non-null counts per column) and `df.sample(5)` (random rows — `head()` can hide weirdness that sorted to the bottom of the file).



## Selecting rows and columns

`df[col]` picks one column, `df[[c1, c2]]` several. `df[mask]` filters rows by a boolean Series. `.loc[rows, cols]` does both by **label** (column names are labels); `.iloc[i, j]` does both by **position**.



In [ ]:
df[df.age > 18]                                  # rows where age > 18
df[(df.age > 18) & (df.city == "Almaty")]        # & not and; each condition in ()
df.loc[df.city == "Almaty", ["age", "income"]]   # those rows, only those columns
df.iloc[:5, :3]                                  # first 5 rows, first 3 columns, by position
df[df.city.isin(["A", "B"])]                     # membership test
df.nlargest(10, "income")                        # top-10 rows by a column

- A filtered result is a copy: `df[df.x > 0]["y"] = 1` may warn (SettingWithCopy) and silently write nothing back. Use `df.loc[df.x > 0, "y"] = 1`.
- `df["a"]` on a missing column raises KeyError; `df.a` fails silently when `a` collides with a method name — brackets are safer.



## Cleaning: NaNs, dtypes, duplicates

Real contest data is dirty. `isna()`/`notna()` find holes, `fillna` patches them, `dropna` deletes rows (`axis=1` deletes columns), `astype` converts dtypes, `drop_duplicates` removes repeats, and the `.str` accessor vectorizes string methods over a whole column.



In [ ]:
df["age"] = df["age"].fillna(df["age"].median())   # median, not mean — robust to outliers
df = df.dropna(subset=["target"])                   # rows missing the label are useless
df["price"] = df["price"].astype("float32")
df = df.drop_duplicates(subset=["user_id"], keep="last")
df["city"] = df["city"].str.strip().str.lower()     # " Almaty " → "almaty" for every row

**Why median for NaNs**: a feature with a fat right tail drags the mean toward its outliers; the median sits at the 50th percentile regardless. Even better in contests: add an `age_missing` indicator column — "was missing" is often itself a signal.

- `isna()` won't catch `"N/A"` or `"?"` strings — scan `.unique()` (or `value_counts`) before trusting the NaN count.
- `dropna()` with no arguments drops any row containing any NaN — on wide tables that can delete most of the data.
- Duplicate rows across the train/validation boundary are **leakage** (information from the answer — or from future data — sneaking into the features) — dedupe **before** splitting.



## Grouping and joining



In [ ]:
df.groupby("city")["income"].agg(["mean", "count"])   # summary table: one row per city
df = df.merge(users, on="user_id", how="left")   # users: a second table keyed by user_id
X = pd.get_dummies(df, columns=["city"])         # one-hot: one 0/1 column per city

Two ways to use a group statistic. `groupby(...).mean()` returns **one row per group** (a summary table). `groupby(...).transform("mean")` returns **one value per original row** — each row gets its own group's mean — so the result lands straight back as a new column:



In [ ]:
df.groupby("user_id")["amount"].mean()               # one row per user
df["user_avg"] = df.groupby("user_id")["amount"].transform("mean")   # same length as df
df.pivot_table(index="city", columns="target", values="income", aggfunc="mean")

> **Note.** groupby + agg/transform is the main engine of feature engineering: per-user averages, per-category counts and per-window statistics become columns with no loops.

- After `merge`, always re-check `len(df)` — a duplicated key on the right side silently multiplies rows.
- `how="inner"` drops unmatched rows; `"left"` keeps them with NaNs. Choose deliberately, then re-check `.isna().mean()`.



### ✏️ Which call shows the share of missing values in each column?

**A.** df.count()
**B.** df.describe()
**C.** df.isna().mean()
**D.** df.dropna()

<details><summary>Answer</summary>

**C.** df.isna().mean()

isna() returns True/False per cell; the mean of booleans is the fraction of True.

</details>

### ✏️ You want every transaction row to get that user's average purchase amount. Which call?

**A.** df.pivot_table(index='user_id', values='amount')
**B.** df.groupby('user_id').agg({'amount': 'mean'})
**C.** df.groupby('user_id')['amount'].transform('mean')
**D.** df.sort_values('amount').groupby('user_id').head(1)

<details><summary>Answer</summary>

**C.** df.groupby('user_id')['amount'].transform('mean')

transform returns a result aligned with the original rows; .mean() returns one row per user.

</details>

### ✏️ df.merge(users, on='uid') doubled the row count. Most likely cause?

**A.** df's uid column has NaNs
**B.** left join is the default
**C.** merge always appends new rows
**D.** users contains duplicate uid values

<details><summary>Answer</summary>

**D.** users contains duplicate uid values

A key matching k rows on the right yields k output rows — dedupe or aggregate the right table first.

</details>

---

**Go deeper**
- [pandas: 10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html)